##**What is LoRA?**

***LoRA = Low-Rank Adaptation***

Definition

**LoRA fine-tunes a large language model by freezing the original weights and training small low-rank matrices that adapt the model to a new task.**

📌 Only a tiny fraction of parameters are trained.

Why LoRA Exists (Problem it Solves)
Full Fine-Tuning Problems

- Very expensive 💰

- Needs huge GPUs

- Slow training

- Causes catastrophic forgetting

👉 LoRA solves all of this




Where LoRA is Applied (Important)

LoRA is usually applied to attention layers:

- Query (Q)

- Key (K)

- Value (V)

- Output (O)

📌 Most common: Q and V only

Why?

Biggest impact

Best efficiency





**LoRA Training Workflow**

- Load pre-trained model

- Freeze all base weights

- Insert LoRA adapters

- Train only LoRA parameters

- Save LoRA weights

- Merge or load adapters during inference



LoRA + QLoRA (Very Important)
QLoRA

Base model in 4-bit

LoRA adapters trained in FP16/BF16

🔥 Fine-tune 65B models on 24GB GPU

**Inference with LoRA**

Two options:

- Merge weights

- Faster inference

- One task

Dynamic adapters

- Swap adapters

- Multiple tasks

##**Step 1: Install Required Libraries**

In [1]:
# !pip install -q transformers datasets peft accelerate evaluate torch

transformers → to load pretrained models and tokenizers like flan-t5-small.

datasets → for Hugging Face datasets like SQuAD.

peft → LoRA / PEFT implementation.

accelerate → automatically moves model/data to GPU, handles mixed precision.

evaluate → built-in evaluation metrics (BLEU, ROUGE, EM, F1).

torch → PyTorch backend.

##**STEP 2: DATASET**

In [2]:
from datasets import load_dataset


dataset = load_dataset("squad")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


##**Step 3: Explore the Dataset (VERY IMPORTANT)**
**3.1 Check dataset splits**

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

**3.2 Look at one training example**

In [4]:
dataset["train"][0]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

In [10]:
# dataset["train"]['answers']

Column([['Saint Bernadette Soubirous'], ['a copper statue of Christ'], ['the Main Building'], ['a Marian place of prayer and reflection'], ['a golden statue of the Virgin Mary']])

##**Step 4: Convert Q&A to Instruction Format**
LLMs work best with instruction-style data.

Format we want:

Instruction: Answer the question using the context
- Context: ...
- Question: ...
- Answer: ..

**4.1 Create formatting function**

In [5]:
def format_qa(example):
  answer = example['answers']['text'][0]
  prompt = f"""Answer the question based on the context.


  Context: {example['context']}
  Question: {example['question']}
  Answer:"""


  return {
  "prompt": prompt,
  "answer": answer
  }

LLMs respond better to instruction-style prompts.

prompt includes context + question, which simulates real-world Q&A.

Return both prompt (input) and answer (label).

This is critical for Seq2Seq training.

**4.2 Apply formatting**

Applies the formatting to every example.

map() is efficient for large datasets.

Now every row has "prompt" and "answer" columns.

In [6]:
formatted_dataset = dataset.map(format_qa)
print(formatted_dataset)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'prompt', 'answer'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'prompt', 'answer'],
        num_rows: 10570
    })
})


In [7]:
formatted_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'prompt', 'answer'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'prompt', 'answer'],
        num_rows: 10570
    })
})

##**Step 5: Load Model & Tokenizer**

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [22]:
print(model.config)

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 1024,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 8,
  "num_heads": 6,
  "num_layers": 8,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": true,
      "max_length": 300,
     

**Step 6: Tokenization (Critical Step)**


Why?

Models only understand tokens, not raw text.

In [11]:
def tokenize(example):
  model_inputs = tokenizer(
  example['prompt'],
  truncation=True,
  padding='max_length',
  max_length=512
  )


  labels = tokenizer(
  example['answer'],
  truncation=True,
  padding='max_length',
  max_length=64
  )


  model_inputs['labels'] = labels['input_ids']
  return model_inputs

Explanation:

truncation=True → removes extra tokens if input is too long.

padding='max_length' → ensures batching works (all sequences same length).

max_length=512 → context + question may be long, but not exceed GPU memory.

max_length=64 → answers are short.

labels → expected output for model to learn.

This ensures input IDs and labels are ready for Trainer.

In [12]:
tokenized_dataset = formatted_dataset.map(tokenize, remove_columns=formatted_dataset['train'].column_names)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Removes original columns → only keep input_ids and labels.

Prepares dataset for HF Trainer.

In [13]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10570
    })
})

##**Step 7: Apply LoRA (PEFT)**
**7.1 Configure LoRA**

In [14]:
from peft import LoraConfig, get_peft_model


lora_config = LoraConfig(
r=8,
lora_alpha=16,
target_modules=["q", "v"],
lora_dropout=0.1,
bias="none",
task_type="SEQ_2_SEQ_LM"
)

r=8 → low-rank update size (small → less GPU memory, big enough to learn).

lora_alpha=16 → scaling factor for stability.

target_modules=["q","v"] → most important attention layers.

lora_dropout=0.1 → prevents overfitting on small data.

bias="none" → keep base model frozen.

task_type="SEQ_2_SEQ_LM" → appropriate for generation tasks.

print_trainable_parameters() → confirms ~1% params trainable.

**7.2 Attach LoRA to model**

In [15]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters().          #📌 You should see ~1% trainable parameters

trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


**Step 8: Training Setup**

In [16]:
from transformers import TrainingArguments, Trainer


training_args = TrainingArguments(
output_dir="./lora-qa",
per_device_train_batch_size=4,
learning_rate=2e-4,
num_train_epochs=1,
logging_steps=50,
save_strategy="epoch",
fp16=True
)


##**Step 9: Train the Model**

In [17]:
trainer = Trainer(
model=model,
args=training_args,
train_dataset=tokenized_dataset['train'].select(range(2000))
)


trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 -


wandb: WARNING Invalid choice
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss
50,0.000000
100,0.000000
150,0.000000
200,0.000000
250,0.000000
300,0.000000
350,0.000000
400,0.000000
450,0.000000
500,0.000000


TrainOutput(global_step=500, training_loss=0.0, metrics={'train_runtime': 144.2981, 'train_samples_per_second': 13.86, 'train_steps_per_second': 3.465, 'total_flos': 373894938624000.0, 'train_loss': 0.0, 'epoch': 1.0})

##**Step 10: Test the Fine-Tuned Model**

In [19]:
def generate_answer(context, question):
  prompt = f"Answer the question based on the context.\n\nContext: {context}\nQuestion: {question}\nAnswer:"
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  outputs = model.generate(**inputs, max_new_tokens=50)
  return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [20]:
example = dataset['validation'][0]
print("Question:", example['question'])
print("Model Answer:", generate_answer(example['context'], example['question']))
print("True Answer:", example['answers']['text'][0])

Question: Which NFL team represented the AFC at Super Bowl 50?
Model Answer: Denver Broncos
True Answer: Denver Broncos


##**Step11.Save Model (LoRA Only)**

In [21]:
model.save_pretrained("lora-qa-adapter")
tokenizer.save_pretrained("lora-qa-adapter")

('lora-qa-adapter/tokenizer_config.json',
 'lora-qa-adapter/special_tokens_map.json',
 'lora-qa-adapter/spiece.model',
 'lora-qa-adapter/added_tokens.json',
 'lora-qa-adapter/tokenizer.json')